In [36]:
import nibabel as nib
from collections import defaultdict
import numpy as np
from brain_connectivity_vizuals import load_mesh_file, create_brain_connectivity_plot, create_modularity_visualization
import pandas as pd
import os
from brain_modularity_pipeline_enhanced_final_v4 import run_enhanced_visualization_pipeline

In [37]:
# # CIMT_rest setup
# study = 'CIMT'
# study_group = study + '/rs'
# ics_path = '../CIMT_data/no_wm/rs/rs_ICA_18c_hptf_ss/rs_ICA_18c_hptf_ss_agg__component_ica_denoised_.nii'
# atlas_path = '../CIMT_data/no_wm/rs/rs_info/rs_01_study_specific_atlas_relabel_no_wm.nii'
# mesh_path = '../CIMT_data/no_wm/brain_filled_3_smoothed.gii'
# roi_coords_path = '../CIMT_data/no_wm/rs/rs_info/resting_atlas_114_mapped_comma.csv'
# hmm_run = 'ICA_14c_no_TDE'
# k_values = list(range(2, 15))

# CIMT_task setup
study = 'CIMT'
study_group = study + '/LLstim'
ics_path = '../CIMT_data/no_wm/LLstim/LLstim_ICA_18c_hptf_ss/LLstim_ICA_18c_hptf_ss_agg__component_ica_denoised_.nii'
atlas_path = '../CIMT_data/no_wm/LLstim/LLstim_info/LLstim_01_study_specific_atlas_relabel_no_wm.nii'
mesh_path = '../CIMT_data/no_wm/brain_filled_3_smoothed.gii'
roi_coords_path = '../CIMT_data/no_wm/LLstim/LLstim_info/atlas_114_mapped_comma.csv'
hmm_run = 'ICA_14c_no_TDE'
k_values = list(range(2, 15))

# # FPI setup
# study_group = 'FPI'
# ics_path = '../FPI_RS_share/no_wm/ICA_13c/ICA_13c_agg__component_ica_.nii'
# atlas_path = '../FPI_RS_share/no_wm/Template/01_study_specific_atlas_relabel_no_wm.nii'
# mesh_path = '../FPI_RS_share/full/Template/MDT_template0.gii'
# roi_coords_path = '../FPI_RS_share/no_wm/FPI_atlas_114_mapped_comma.csv'
# hmm_run = 'ICA_13c_no_TDE'
# k_values = list(range(11, 12))

all_covs, state_plots_paths, modularity_plots_paths = {}, {}, {}
for k in k_values:
    if not os.path.exists(f'{study_group}/results/{hmm_run}/{k}_states/inf_params'):
        raise FileNotFoundError(f"Inference parameters for k={k} states not found")
        
    covs = np.load(open(f'{study_group}/results/{hmm_run}/{k}_states/inf_params/covs_{k}_states.npy', 'rb'))
    print(covs.shape)
    all_covs[k] = covs

    state_plots_path = f'{study_group}/results/{hmm_run}/{k}_states/brain_state_plots'
    os.makedirs(state_plots_path, exist_ok=True)
    state_plots_paths[k] = state_plots_path

    modularity_plots_path = f'{study_group}/results/{hmm_run}/{k}_states/modularity_plots'
    os.makedirs(modularity_plots_path, exist_ok=True)
    modularity_plots_paths[k] = modularity_plots_path

(2, 14, 14)
(3, 14, 14)
(4, 14, 14)
(5, 14, 14)
(6, 14, 14)
(7, 14, 14)
(8, 14, 14)
(9, 14, 14)
(10, 14, 14)
(11, 14, 14)
(12, 14, 14)
(13, 14, 14)
(14, 14, 14)


In [38]:
ics = nib.load(ics_path)
ics_data = ics.get_fdata()
ics_data.shape

(96, 96, 25, 14)

In [39]:
atlas = nib.load(atlas_path)
atlas_data = atlas.get_fdata()
atlas_data.shape

(96, 96, 25)

# Create n_ROI x n_IC Matrix

In [40]:
roi_to_ic = defaultdict(lambda: defaultdict(list)) # roi_idx: {ic: loadings for each coordinate of the roi}
for i in range(ics_data.shape[0]):
    for j in range(ics_data.shape[1]):
        for k in range(ics_data.shape[2]):
            for ic in range(ics_data.shape[3]):
                idx = int(atlas_data[i, j, k])
                if idx > 0:
                    roi_to_ic[idx][ic + 1].append(ics_data[i, j, k, ic])

In [41]:
roi_to_ic[1]

defaultdict(list,
            {1: [np.float64(-1.2676212787628174),
              np.float64(-1.4273436069488525),
              np.float64(-0.3156186640262604),
              np.float64(-0.36106473207473755),
              np.float64(-0.6865568161010742),
              np.float64(0.0560142882168293),
              np.float64(-1.29816734790802),
              np.float64(-1.6765402555465698),
              np.float64(-0.6822714805603027),
              np.float64(-0.35633111000061035),
              np.float64(-1.0258185863494873),
              np.float64(-1.537592887878418),
              np.float64(-0.507245659828186),
              np.float64(-0.39146071672439575),
              np.float64(-0.9879043698310852),
              np.float64(-1.4632269144058228),
              np.float64(-0.3816013038158417),
              np.float64(-0.4343782663345337),
              np.float64(-0.43342190980911255),
              np.float64(-0.5222105979919434),
              np.float64(-0.439111620187

In [42]:
A = np.zeros((len(roi_to_ic), ics_data.shape[3]))
for roi in sorted(roi_to_ic.keys()):
    for ic in sorted(roi_to_ic[roi].keys()):
        A[roi - 1, ic - 1] = np.mean(roi_to_ic[roi][ic])

A.shape

(114, 14)

# Create n_ROI x n_ROI Correlation Matrix

In [43]:
all_roi_corrs = {}
for k in k_values:
    roi_corrs = []
    for cov in all_covs[k]:
        roi_cov = A @ cov @ A.T
        roi_cov = 0.5 * (roi_cov + roi_cov.T) # enforce symmetry just in case of floating point error
        
        std = np.sqrt(np.diag(roi_cov))
        std_safe = std.copy()
        std_safe[std_safe == 0] = 1
        
        denom = np.outer(std_safe, std_safe)
        roi_corr = roi_cov / denom
        
        std_zeros = (std == 0)
        roi_corr[std_zeros, :] = 0
        roi_corr[:, std_zeros] = 0
        roi_corrs.append(roi_corr)

    print(np.array(roi_corrs).shape)
    all_roi_corrs[k] = roi_corrs

(2, 114, 114)
(3, 114, 114)
(4, 114, 114)
(5, 114, 114)
(6, 114, 114)
(7, 114, 114)
(8, 114, 114)
(9, 114, 114)
(10, 114, 114)
(11, 114, 114)
(12, 114, 114)
(13, 114, 114)
(14, 114, 114)


# Make Plots

In [44]:
vertices, faces = load_mesh_file(mesh_path)

Loading mesh from: ../CIMT_data/no_wm/brain_filled_3_smoothed.gii
  Found faces array with shape: (18857, 3)
  Found vertices array with shape: (9510, 3)
  Successfully loaded mesh: 9510 vertices, 18857 faces


In [45]:
roi_coords_df = pd.read_csv(roi_coords_path)
roi_coords_df.head(20)

,mapped_index,original_index,roi_name,cog_x,cog_y,cog_z,cog_voxel_i,cog_voxel_j,cog_voxel_k
0,1,1,Acumbens_left,-15.355556,66.017901,-20.083333,79.284444,34.933333,142.814321
1,2,2,AID_left,-45.765769,64.444362,-7.542770,103.612615,44.965784,141.555489
2,3,3,AIP_left,-61.894410,29.911610,-21.016484,116.515528,34.186813,113.929288
3,4,4,AIV_left,-38.613395,69.366538,-9.036509,97.890716,43.770793,145.493230
4,5,5,Amygdala_left,-40.170219,15.355125,-37.531890,99.136175,20.974488,102.284100
5,6,8,Au1_left,-67.019196,-0.812283,4.944829,120.615357,54.955864,89.350173
6,7,9,AUD_left,-62.842693,3.178845,12.781300,117.274155,61.225040,92.543076
7,8,10,AuV_left,-69.197761,-2.039801,-6.915423,122.358209,45.467662,88.368159
8,9,11,Brainstem_left,-18.433435,-57.047145,-28.892489,81.746748,27.886009,44.362284
9,10,12,Cent_Gray_left,-7.648063,-22.936525,-0.421196,73.118450,50.663043,71.650780


In [46]:
# Generate brain state plots
all_roi_corrs_thr = {}
for k in k_values:
    roi_corrs_thr = []
    for i in range(len(all_roi_corrs[k])):
        # Visualize top 2.5% of correlations for each state
        R = roi_corrs[i]
        iu = np.triu_indices_from(R, k=1)
        vals = np.abs(R[iu]) # (n_ROI * (n_ROI - 1) / 2,)
        thr = np.percentile(vals, 97.5)
        R_thr = np.zeros_like(R)
        keep = vals >= thr
        R_thr[iu[0][keep], iu[1][keep]] = R[iu[0][keep], iu[1][keep]]
        R_thr = R_thr + R_thr.T # symmetrize
        roi_corrs_thr.append(R_thr)
        # create_brain_connectivity_plot(
        #     vertices=vertices, 
        #     faces=faces, 
        #     roi_coords_df=roi_coords_df, 
        #     connectivity_matrix=R_thr,
        #     plot_title=f"{study} HMM k={k}, State {i + 1}",
        #     save_path=os.path.join(state_plots_paths[k], f'state_{i + 1}.html'),
        #     node_size=8,
        #     node_color='purple',
        #     node_border_color='magenta',
        #     pos_edge_color='red',
        #     neg_edge_color='blue',
        #     edge_width=5,
        #     edge_threshold=0.0,
        #     mesh_color='lightgray',
        #     mesh_opacity=0.5,
        #     label_font_size=8,
        #     fast_render=False,
        # )

    all_roi_corrs_thr[k] = roi_corrs_thr

In [47]:
for k in k_values:
    np.save(
        f'{study_group}/results/{hmm_run}/{k}_states/inf_params/all_corrs_{k}_states.npy', 
        np.array(all_roi_corrs[k]),
    )
    
    np.savez(
        f'{study_group}/results/{hmm_run}/{k}_states/inf_params/all_corrs_{k}_states.npz', 
        centroids=np.array(all_roi_corrs[k]),
    )

In [48]:
for k in k_values:
    print(np.array(all_roi_corrs[k]).shape)

(2, 114, 114)
(3, 114, 114)
(4, 114, 114)
(5, 114, 114)
(6, 114, 114)
(7, 114, 114)
(8, 114, 114)
(9, 114, 114)
(10, 114, 114)
(11, 114, 114)
(12, 114, 114)
(13, 114, 114)
(14, 114, 114)


***Everything below this requires OMST filtering + expanded modularity pipeline***

# Make Plots Using brain_modularity_pipeline_enhanced_final_v4

In [49]:
if not os.path.exists(f'{study}/omst_filter_matrices'):
        raise FileNotFoundError(f"OMST-filtered matrices not found for {study}. Please run OMST filtering first")
    
matrix_paths = {}
for k in k_values:
    if study == 'CIMT':
        if study_group == 'CIMT/rs':
            matrix_paths[k] = f'CIMT/omst_filter_matrices/resting_matrices/k{k}_strategy_A_positive.npy'
        elif study_group == 'CIMT/LLstim':
            matrix_paths[k] = f'CIMT/omst_filter_matrices/stim_matrices/k{k}_strategy_A_positive.npy'

if not os.path.exists(f'{study_group}/results/{hmm_run}/modularity_significance_analysis_omst_pos'):
    raise FileNotFoundError(f"Modularity analysis results not found for {study_group}. Please run the expanded modularity pipeline first")
        
modularity_analysis_path = f'{study_group}/results/{hmm_run}/modularity_significance_analysis_omst_pos'

In [50]:
modularity_pvalue_fdr_complete = pd.read_csv(modularity_analysis_path + '/comprehensive_analysis/network_metrics_summary.csv')
print(modularity_pvalue_fdr_complete.to_string(index=False))

 k  state  mean_pc   std_pc        mean_z  std_z  n_hubs  n_connectors  n_significant_pc
 2      0 0.295734 0.186494 -1.363432e-16    1.0       0             2               110
 2      1 0.255353 0.183688 -9.738798e-18    1.0       1             2               113
 3      0 0.266320 0.185611  1.129701e-16    1.0       0             3               111
 3      1 0.291062 0.184851 -1.830894e-16    1.0       0             2               110
 3      2 0.234509 0.203311  1.752984e-17    1.0       0             2               109
 4      0 0.187660 0.208322  2.031148e-16    1.0       0             1                97
 4      1 0.265873 0.189343 -5.064175e-17    1.0       0             1               111
 4      2 0.253711 0.196843 -7.791039e-18    1.0       0             0               110
 4      3 0.318410 0.186037  1.319607e-16    1.0       0             4               110
 5      0 0.255994 0.201833  1.480297e-16    1.0       0             3               107
 5      1 0.259091 0.

In [51]:
# Search for hubs
def n_states_with_hubs(group):
    return (group['n_hubs'] > 0).sum()

def states_with_hubs(group):
    states = (group[group['n_hubs'] > 0]['state'] + 1).tolist()
    return ", ".join(map(str, states))

def total_n_hubs(group):
    return group['n_hubs'].sum()

grouped_df = modularity_pvalue_fdr_complete.groupby('k').apply(
    lambda group: pd.Series({
        'n_states_with_hubs': n_states_with_hubs(group),
        'states_with_hubs': states_with_hubs(group),
        'total_n_hubs': total_n_hubs(group)
    })
).reset_index()
print(grouped_df.to_string(index=False)) 

 k  n_states_with_hubs states_with_hubs  total_n_hubs
 2                   1                2             1
 3                   0                              0
 4                   0                              0
 5                   1                2             1
 6                   2             4, 6             2
 7                   1                2             1
 8                   2             6, 8             2
 9                   1                1             2
10                   0                              0
11                   3          3, 4, 5             4
12                   3         7, 9, 12             3
13                   4     3, 5, 11, 12             5
14                   2            3, 10             2


In [ ]:
for k in k_values:
    run_enhanced_visualization_pipeline(
        matrix_path=matrix_paths[k],
        netneurotools_results_dir=modularity_analysis_path, 
        mesh_file=mesh_path,
        roi_coords_file=roi_coords_path,
        output_dir=modularity_plots_paths[k] + '/enhanced',
        k_value=k,
    )

ENHANCED BRAIN MODULARITY VISUALIZATION PIPELINE - VERSION 4
With Interactive Camera Controls and Multiple Views
Output directory: CIMT/LLstim/results/ICA_14c_no_TDE/2_states/modularity_plots/enhanced
Visualization types: ['all', 'intra', 'inter', 'nodes_only', 'significant_only']
Node sizing modes: ['pc', 'zscore', 'both']
Camera views: ['oblique']
Interactive panel: Enabled
Border width: 6 pixels (enhanced visibility)

1. Loading brain mesh and ROI data...
Loading mesh from: ../CIMT_data/no_wm/brain_filled_3_smoothed.gii
  Found faces array with shape: (18857, 3)
  Found vertices array with shape: (9510, 3)
  Successfully loaded mesh: 9510 vertices, 18857 faces
   Loaded 114 ROIs
   Mesh vertices: (9510, 3)
   Mesh faces: (18857, 3)

2. Loading connectivity matrices...
   Loaded matrices: shape (2, 114, 114)

3. Loading NetNeurotools results...
NetNeurotools Loader initialized with base: CIMT/LLstim/results/ICA_14c_no_TDE/modularity_significance_analysis_omst_pos
   Loading summary s